# Lecture 3 · Notebook 1 — N-gram language models: counting, spinning the wheel, smoothing, perplexity
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture03/lecture03_1_ngram_language_models.ipynb)

**Companion to [Chapters 3–5 of the video lecture](https://nlp-course-cuhksz.github.io/lecture03.html#p3-s35)** (slides 34–82). A language model gives a probability to every
piece of text. The simplest way to get one is to **count**. In this notebook you will:

1. count unigrams and bigrams in the Alice passage from the slides and reproduce the numbers on slides 27 and 38,
2. replay the **spin-the-wheel** generation from slides 50–60 (bigram and trigram), with the same random numbers,
3. train 1- to 4-gram models on *tiny Shakespeare* and see text get more fluent but also more copied,
4. measure **sparsity**, watch MLE give a test sentence probability zero, and fix it with **smoothing**, and
5. compute **perplexity** with log-probabilities, the way you will in the assignment.

Pure Python, runs on CPU in about a minute. Nothing to install.

## 1 · Counting the Alice passage (slide 27)
We split the text into words and punctuation marks; every token is "a shape in a bag". Our simple tokenizer finds 67 tokens
(the slide counts 66 — tokenizers differ by a punctuation mark here and there), so the probabilities below are *k*/67.

In [ ]:
import re
from collections import Counter, defaultdict

alice = ("Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: "
         "once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations "
         "in it, 'and what is the use of a book,' thought Alice 'without pictures or conversation?'")
tokens = re.findall(r"\w+|[^\w\s]", alice)
N = len(tokens)
uni = Counter(tokens)
print(N, "tokens")
for w in ["of", "Alice", "was", "to", "her", "sister", ",", "'"]:
    print(f"P({w!r}) = {uni[w]}/{N} = {uni[w] / N:.3f}")

## 2 · A bigram model by counting (slides 38 and 40)
Maximum-likelihood estimation is just relative frequency: $P(w \mid v) = \dfrac{c(v\,w)}{c(v)}$.

In [ ]:
def ngram_table(tokens, n):
    """context (tuple of n-1 words) -> Counter of next words, in order of first appearance"""
    table = defaultdict(Counter)
    for i in range(len(tokens) - n + 1):
        table[tuple(tokens[i:i + n - 1])][tokens[i + n - 1]] += 1
    return table

def dist(table, context):
    c = table[tuple(context)]
    total = sum(c.values())
    return {w: k / total for w, k in c.items()}

bigram = ngram_table(tokens, 2)
for v in ["the", "tired", "was", "book", "her", "sister"]:
    print(f"P(. | {v!r}) =", {w: round(p, 3) for w, p in dist(bigram, [v]).items()})

## 3 · Spin the wheel (slides 49–60)
To sample from a distribution, cut the interval [0, 1) into one slice per word, draw a random number *r*, and return the word whose slice
contains *r*. We first replay the lecture with **the same r values** that appear on the slides, then let Python spin.

In [ ]:
def spin(d, r):
    """return the word whose interval [lo, hi) contains r"""
    lo = 0.0
    for w, p in d.items():
        if r < lo + p:
            return w, lo, lo + p
        lo += p
    return w, lo - p, 1.0

# bigram run from slides 50-54
context, out = ["the"], ["the"]
for r in [0.52, 0.31, 0.74, 0.68, 0.87]:
    w, lo, hi = spin(dist(bigram, context[-1:]), r)
    print(f"after {context[-1]!r:10s} r = {r:.2f} falls in [{lo:.2f}, {hi:.2f})  -> {w!r}")
    context.append(w); out.append(w)
print("bigram  :", " ".join(out))

# trigram run from slides 55-60: only the wheel after "her sister" has two slices
trigram = ngram_table(tokens, 3)
out = ["the", "book"]
for step in range(6):
    d = dist(trigram, out[-2:])
    w, lo, hi = spin(d, 0.29)
    print(f"after {' '.join(out[-2:])!r:16s} {len(d)} slice(s) -> {w!r}")
    out.append(w)
print("trigram :", " ".join(out))

Now spin with real random numbers. Run the cell a few times: the bigram model stitches pieces of the passage together, the trigram model mostly copies it.

In [ ]:
import random
def generate(table, n, start, length=12, rng=random):
    out = list(start)
    for _ in range(length):
        d = dist(table, out[-(n - 1):])
        if not d:
            break
        out.append(spin(d, rng.random())[0])
    return " ".join(out)

for _ in range(3):
    print("bigram :", generate(bigram, 2, ["the"]))
for _ in range(3):
    print("trigram:", generate(trigram, 3, ["the", "book"]))

## 4 · A real corpus: tiny Shakespeare
About 1 MB of Shakespeare (Karpathy's *tiny Shakespeare*, a subset of the plays; the numbers on slide 64 are for the complete works).
We lowercase, split into words and punctuation, treat every line as a sentence, and hold out 10% of the lines as a test set.
Another 5% is a **dev** set for tuning. Rare words (seen once) become `<unk>`, and each line ends with `</s>` (slides 42–46).

In [ ]:
import os, re, math, random, urllib.request
from collections import Counter, defaultdict

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("tinyshakespeare.txt"):
    urllib.request.urlretrieve(URL, "tinyshakespeare.txt")
raw = open("tinyshakespeare.txt", encoding="utf-8").read()

def tokenize(line):
    return re.findall(r"[a-z]+(?:'[a-z]+)?|[^\sa-z]", line.lower())

lines = [tokenize(l) for l in raw.split("\n")]
lines = [l for l in lines if l]                      # every non-empty line is one "sentence"
random.Random(0).shuffle(lines)
n_test, n_dev = len(lines) // 10, len(lines) // 20
test_lines, dev_lines, train_lines = lines[:n_test], lines[n_test:n_test + n_dev], lines[n_test + n_dev:]

# vocabulary: words seen at least twice in training; everything else becomes <unk> (slide 45)
wc = Counter(w for l in train_lines for w in l)
vocab = {w for w, c in wc.items() if c >= 2} | {"<unk>", "</s>"}
def unkify(l):
    return [w if w in vocab else "<unk>" for w in l]
train = [unkify(l) for l in train_lines]
test = [unkify(l) for l in test_lines]
dev = [unkify(l) for l in dev_lines]       # held-out lines for tuning (never used for counting or training)
print(f"{len(train)} training lines, {sum(map(len, train)):,} training tokens, {len(dev)} dev lines, {len(test)} test lines")
print(f"vocabulary: {len(vocab):,} types (incl. <unk>, </s>)")
print("example line:", train[0])

### Sparsity (slide 64)
How many of the possible n-grams does the corpus actually contain?

In [ ]:
BOS, EOS = "<s>", "</s>"
def padded(line, n):
    return [BOS] * (n - 1) + line + [EOS]

def count_ngrams(lines, n):
    c = Counter()
    for l in lines:
        p = padded(l, n)
        for i in range(len(p) - n + 1):
            c[tuple(p[i:i + n])] += 1
    return c

counts = {n: count_ngrams(train, n) for n in range(1, 5)}
V = len(vocab)
for n in range(1, 5):
    seen = len(counts[n])
    print(f"{n}-grams: {seen:>9,} types seen  of  V^{n} = {float(V) ** n:.2e} possible  ->  fraction seen {seen / V ** n:.1e}")

### Generating Shakespeare (slides 61–62)
Larger *n* gives more fluent text. But check how much of each generated line is **copied**: we measure what fraction of its 4-grams
also appear in the training data.

In [ ]:
def build(lines, n):
    t = defaultdict(Counter)
    for l in lines:
        p = padded(l, n)
        for i in range(len(p) - n + 1):
            t[tuple(p[i:i + n - 1])][p[i + n - 1]] += 1
    return t

tables = {n: build(train, n) for n in range(1, 5)}
train4 = set(counts[4])
rng = random.Random(1)

def sample_line(n, max_len=25):
    while True:                                   # resample if the line ends almost immediately
        s = _sample(n, max_len)
        if len(s) >= 4:
            return s

def _sample(n, max_len):
    out = [BOS] * (n - 1)
    while len(out) < max_len + n - 1:
        c = tables[n][tuple(out[len(out) - (n - 1):]) if n > 1 else ()]
        w = rng.choices(list(c), weights=list(c.values()))[0]
        if w == EOS:
            break
        out.append(w)
    return out[n - 1:]

for n in range(1, 5):
    for _ in range(2):
        s = sample_line(n)
        grams = [tuple(s[i:i + 4]) for i in range(len(s) - 3)]
        copied = sum(g in train4 for g in grams) / max(1, len(grams))
        print(f"{n}-gram | {copied:4.0%} of 4-grams copied | {' '.join(s)}")

## 5 · MLE gives unseen events probability zero (slide 65)
Score a test line with the unsmoothed (MLE) bigram model. One unseen bigram is enough to make the whole line impossible.

In [ ]:
def mle_prob(line, n):
    p, prob = padded(line, n), 1.0
    for i in range(n - 1, len(p)):
        ctx = tuple(p[i - n + 1:i])
        c = tables[n][ctx]
        prob *= c[p[i]] / sum(c.values()) if c else 0.0
    return prob

zero = sum(mle_prob(l, 2) == 0.0 for l in test)
print(f"{zero / len(test):.1%} of test lines get probability 0 under the MLE bigram model")
print("example:", " ".join(next(l for l in test if mle_prob(l, 2) == 0.0)))

unseen = Counter()
for n in (2, 3):
    tot = 0
    for l in test:
        p = padded(l, n)
        for i in range(len(p) - n + 1):
            tot += 1
            unseen[n] += tuple(p[i:i + n]) not in counts[n]
    print(f"{unseen[n] / tot:.1%} of the {n}-grams in the test set never occur in training")

## 6 · Smoothing and perplexity (slides 66–73)
**Add-k smoothing:** $P(w\mid h) = \dfrac{c(h\,w) + k}{c(h) + kV}$.
**Perplexity:** $PP = \exp\big(-\tfrac{1}{N}\sum_i \log P(w_i \mid h_i)\big)$, computed with **logs** to avoid underflow (slide 73).
First, see underflow happen:

In [ ]:
probs = [0.01] * 400                           # 400 tokens, each with probability 0.01
prod = 1.0
for p in probs:
    prod *= p
print("product of probabilities:", prod)       # underflows to 0.0
print("sum of log-probabilities:", sum(math.log(p) for p in probs))

In [ ]:
Vp = len(vocab)            # predicted vocabulary: all words + </s>  (<s> is never predicted)

def perplexity(logp_fn, n, lines):
    total, N = 0.0, 0
    for l in lines:
        p = padded(l, n)
        for i in range(n - 1, len(p)):
            total += logp_fn(tuple(p[i - n + 1:i]), p[i])
            N += 1
    return math.exp(-total / N)

# cache context totals so the loops are fast
for n in tables:
    for ctx, c in tables[n].items():
        c.total = sum(c.values())
def logp_addk(n, ctx, w, k):
    c = tables[n].get(ctx)
    cw, ct = (c[w], c.total) if c else (0, 0)
    return math.log((cw + k) / (ct + k * Vp))

print(f"{'model':10s} " + " ".join(f"{'k=' + str(k):>9s}" for k in [1, 0.1, 0.01, 0.001]))
for n in (1, 2, 3):
    row = [perplexity(lambda ctx, w: logp_addk(n, ctx, w, k), n, test) for k in [1, 0.1, 0.01, 0.001]]
    print(f"{n}-gram     " + " ".join(f"{x:9.1f}" for x in row))

Two lessons from this table, both on the slides. Add-**one** is a poor choice for bigrams and trigrams: it hands most of the probability
mass to the huge number of unseen n-grams (slide 68), so it can be *worse* than a unigram model. A much smaller *k* helps a lot.

### Linear interpolation (slide 67)
Mix trigram, bigram and unigram estimates: $P = \lambda_3 P_3 + \lambda_2 P_2 + \lambda_1 P_1$. We pick the λ's on the
**dev** lines, never on the test set.

In [ ]:
def p_mle(n, ctx, w):
    c = tables[n].get(ctx)
    return c[w] / c.total if c and c.total else 0.0

def logp_interp(ctx3, w, lams):
    l3, l2, l1 = lams
    p1 = (tables[1][()][w] + 1) / (tables[1][()].total + Vp)          # add-one unigram: never zero
    return math.log(l3 * p_mle(3, ctx3, w) + l2 * p_mle(2, ctx3[1:], w) + l1 * p1)

grid = [(a, b, round(1 - a - b, 2)) for a in [0.1, 0.2, 0.3, 0.4, 0.5] for b in [0.2, 0.3, 0.4, 0.5] if a + b < 1]
best = min(grid, key=lambda lams: perplexity(lambda ctx, w: logp_interp(ctx, w, lams), 3, dev))
print("chosen lambdas (3-gram, 2-gram, 1-gram):", best)
print(f"interpolated trigram test perplexity: {perplexity(lambda ctx, w: logp_interp(ctx, w, best), 3, test):.1f}")

A perplexity of about 100–200 means the model is, on average, as unsure as if it chose uniformly among that many words (slide 72).
Notebook 2 trains a small **neural** LM on exactly the same data and compares perplexities.

## Exercises
1. Change the vocabulary threshold from `c >= 2` to `c >= 5`. Perplexity drops — is the model really better? (Slide 72: you can only
   compare perplexities with the **same vocabulary**.)
2. Implement **backoff** (slide 87): if the trigram context was never seen, use the bigram; if that was never seen, the unigram.
   Compare its perplexity with interpolation.
3. Tune the add-k constant *k* for the bigram model on the **dev** lines (try values between 0.0005 and 0.05), then report test
   perplexity once. Why must you not tune on the test set?
4. Generate 20 lines from the 4-gram model and search the corpus (`raw.lower()`) for each one. How many are verbatim copies?

**Next:** [Notebook 2 — a fixed-window neural language model]({colab(N2)}) · **Back to the [video lecture]({SITE})**